# Week 19: MLOps - Versioning and Experiment Tracking

In Week 14 you trained a fraud classifier and saved it to `./fraud-classifier` with no record of which hyperparameters you used. In Week 18 you measured your RAG pipeline with RAGAS and got scores you cannot reproduce. This week you fix both problems - and then wire the versioned model back into the Week 18 supervisor as a new tool.

## Learning objectives

By the end of this session you will be able to:

1. Read a versioned Delta table from Databricks and stage training data in S3 for SageMaker.
2. Track experiment runs (parameters, metrics, artifacts) in SageMaker managed MLflow.
3. Submit a SageMaker Training Job that fine-tunes DistilBERT on fraud data.
4. Register the trained model in the SageMaker Model Registry and promote it to Approved.
5. Deploy the registered model behind a SageMaker endpoint and call it from a Strands tool inside the Week 18 supervisor.

## Prerequisites

- Week 14 (DistilBERT fine-tuning)
- Week 18 (`week18_supervisor` with RAGAS evaluation)
- Databricks workspace access with `aws-course-creds` secret scope

## Environment Setup

**Platform**: Azure Databricks (Runtime 15.4 LTS ML or later).

The next code cell uses `%pip install` (a Databricks magic) to install any libraries the cluster does not already have. After it finishes, Databricks may prompt you to restart the Python kernel - do that, then re-run from the top.

**Pre-installed cluster libraries** (instructor confirms before class):

- `boto3>=1.35`
- `sagemaker>=2.230,<3` (v3 breaks `from sagemaker import get_execution_role`)
- `sagemaker-mlflow>=0.1.0` (plugin so MLflow can talk to the managed tracking server)
- `mlflow>=2.13`
- `strands-agents>=1.37,<2`
- `strands-agents-tools>=0.2`
- `langchain-aws>=0.2`

**Secret scope**: `aws-course-creds` (instructor provisions; keys listed in Cell 4).

Run the verification cell below to confirm everything imports.

In [ ]:
# Install any missing libraries (safe to re-run; Databricks may ask you to restart the kernel afterwards)
%pip install --quiet "boto3>=1.35" "sagemaker>=2.230,<3" "sagemaker-mlflow>=0.1.0" "mlflow>=2.13" "strands-agents>=1.37,<2" "strands-agents-tools>=0.2" "langchain-aws>=0.2"

# Standard library
import os
import json
import time
import tarfile
import io
from datetime import datetime

# Third-party
import boto3
import pandas as pd
import mlflow
from importlib.metadata import version

# Verify versions (no __version__ access - use importlib.metadata per house style)
for pkg in ["boto3", "sagemaker", "mlflow", "sagemaker-mlflow", "strands-agents"]:
    try:
        print(f"{pkg:25s} {version(pkg)}")
    except Exception as e:
        print(f"{pkg:25s} NOT INSTALLED ({e})")

In [ ]:
# Pull AWS credentials from the Databricks secret scope. NEVER hard-code.
AWS_ACCESS_KEY_ID     = dbutils.secrets.get(scope="aws-course-creds", key="aws-access-key-id")
AWS_SECRET_ACCESS_KEY = dbutils.secrets.get(scope="aws-course-creds", key="aws-secret-access-key")
AWS_SESSION_TOKEN     = dbutils.secrets.get(scope="aws-course-creds", key="aws-session-token")
AWS_REGION            = "us-east-1"

SAGEMAKER_ROLE_ARN    = dbutils.secrets.get(scope="aws-course-creds", key="sagemaker-execution-role-arn")
MLFLOW_TRACKING_ARN   = dbutils.secrets.get(scope="aws-course-creds", key="mlflow-tracking-server-arn")

# Export to env for boto3 / sagemaker SDK pickup
os.environ["AWS_ACCESS_KEY_ID"]     = AWS_ACCESS_KEY_ID
os.environ["AWS_SECRET_ACCESS_KEY"] = AWS_SECRET_ACCESS_KEY
os.environ["AWS_SESSION_TOKEN"]     = AWS_SESSION_TOKEN
os.environ["AWS_REGION"]            = AWS_REGION
os.environ["AWS_DEFAULT_REGION"]    = AWS_REGION

# boto3 clients used throughout the notebook
session            = boto3.Session(region_name=AWS_REGION)
s3_client          = session.client("s3")
sagemaker_client   = session.client("sagemaker")
sagemaker_runtime  = session.client("sagemaker-runtime")
sts_client         = session.client("sts")

print("Caller identity:", sts_client.get_caller_identity()["Arn"])

In [ ]:
# Pre-flight probes - fail loud before any real work.
S3_BUCKET = "bread-academy-week19-shared"
S3_PREFIX = f"students/{sts_client.get_caller_identity()['UserId'][:8]}"

# 1) S3 access
try:
    s3_client.head_bucket(Bucket=S3_BUCKET)
    print(f"S3 OK: s3://{S3_BUCKET}")
except Exception as e:
    print(f"S3 FAIL: {e}\nAsk your instructor to grant access to {S3_BUCKET}.")
    raise

# 2) SageMaker access
try:
    sagemaker_client.list_training_jobs(MaxResults=1)
    print("SageMaker OK")
except Exception as e:
    print(f"SageMaker FAIL: {e}")
    raise

# 3) Managed MLflow tracking server
try:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_ARN)
    experiments = mlflow.search_experiments(max_results=1)
    print(f"MLflow OK: tracking_uri={mlflow.get_tracking_uri()}")
except Exception as e:
    print(f"MLflow FAIL: {e}\nAsk your instructor to confirm the MLflow tracking server is running.")
    raise

## Topic 1: Versioned data prep with Spark + Delta

Before you can track an experiment, you need a frozen snapshot of the training data. Delta Lake gives you that for free: every read of `bread_academy.course_data.fraud_transactions` carries a version number, and you can reproduce the exact rows months later with `VERSION AS OF`.

You will:

1. Read the fraud transactions Delta table.
2. Capture its current version (this is the data version you will log to MLflow).
3. Split train/test and stage CSVs in S3 so SageMaker can read them.

In [ ]:
# DEMO: read the pre-loaded Delta table from Unity Catalog
df = spark.read.table("bread_academy.course_data.fraud_transactions")

# Capture the current Delta table version - this is what makes the run reproducible.
# Six months from now, "VERSION AS OF {DATA_VERSION}" returns the exact same rows.
history = spark.sql("DESCRIBE HISTORY bread_academy.course_data.fraud_transactions LIMIT 1").collect()
DATA_VERSION = history[0]["version"]

print(f"Rows: {df.count()}, columns: {df.columns}")
print(f"Delta version: {DATA_VERSION}")
display(df.limit(5))

In [ ]:
# DEMO: train/test split and stage as CSV in S3 so SageMaker Training can read it
from pyspark.sql.functions import col

# Use the same schema Week 14 expected: text column + binary label
prepared = df.select(
    col("description").alias("text"),
    col("is_fraud").cast("int").alias("label"),
)

train_df, test_df = prepared.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_df.count()}, Test: {test_df.count()}")

# Stage as CSV in S3. coalesce(1) so SageMaker sees a single file per channel.
# The path includes v{DATA_VERSION} so we can always tell which Delta snapshot produced these files.
TRAIN_S3 = f"s3://{S3_BUCKET}/{S3_PREFIX}/data/v{DATA_VERSION}/train"
TEST_S3  = f"s3://{S3_BUCKET}/{S3_PREFIX}/data/v{DATA_VERSION}/test"

(train_df.coalesce(1)
    .write.mode("overwrite").option("header", True)
    .csv(TRAIN_S3))
(test_df.coalesce(1)
    .write.mode("overwrite").option("header", True)
    .csv(TEST_S3))

print(f"Train -> {TRAIN_S3}")
print(f"Test  -> {TEST_S3}")

### Think About It

You just wrote train/test CSVs to a path that includes the Delta version number (`v{DATA_VERSION}`). Why does this matter for reproducibility? If a colleague tries to reproduce your run six months from now and the Delta table has been updated 50 times, what part of the path lets them get back to YOUR exact rows? What would you lose if you skipped the version in the path?

## Topic 2: MLflow experiment tracking

SageMaker hosts a managed MLflow tracking server for you. You connect to it by setting `mlflow.set_tracking_uri(<server-arn>)` after installing the `sagemaker-mlflow` plugin. Every `mlflow.start_run()` writes parameters, metrics, and artifacts to S3 with full lineage.

We will start by logging your Week 18 RAGAS scores as the BASELINE run. That way next week, when you tune the RAG pipeline, you can answer the question Week 18 left open: "is this better than last week?"

In [ ]:
# DEMO: set the experiment and log the Week 18 RAGAS baseline run
# Same tracking URI was set in the probe; making it explicit here for clarity
mlflow.set_tracking_uri(MLFLOW_TRACKING_ARN)

EXPERIMENT_NAME = f"week19-fraud-mlops-{sts_client.get_caller_identity()['UserId'][:8]}"
mlflow.set_experiment(EXPERIMENT_NAME)
print(f"Experiment: {EXPERIMENT_NAME}")

# These are the RAGAS scores from Week 18 - hard-coded here so every student
# has a baseline to compare against. In real life these would be loaded from
# a CSV you saved at the end of Week 18.
week18_ragas = {
    "faithfulness": 0.82,
    "answer_relevancy": 0.79,
    "context_precision": 0.74,
}
week18_pipeline_params = {
    "retriever": "bedrock_kb_FARSQGTONR",
    "reranker": "cohere.rerank-v3-5:0",
    "top_k_retrieve": 10,
    "top_k_rerank": 3,
    "llm": "us.anthropic.claude-3-haiku-20240307-v1:0",
}

with mlflow.start_run(run_name="week18-ragas-baseline") as run:
    mlflow.set_tag("source_week", "18")
    mlflow.set_tag("pipeline_type", "rag")
    mlflow.log_params(week18_pipeline_params)
    mlflow.log_metrics(week18_ragas)

    # Log the comparison DataFrame as an artifact
    pd.DataFrame([week18_ragas]).to_csv("/tmp/week18_ragas.csv", index=False)
    mlflow.log_artifact("/tmp/week18_ragas.csv")

    BASELINE_RUN_ID = run.info.run_id
    print(f"Logged baseline run: {BASELINE_RUN_ID}")

In [ ]:
# DEMO: open the managed MLflow UI in a browser. The URL is short-lived (~5 min).
url_resp = sagemaker_client.create_presigned_mlflow_tracking_server_url(
    TrackingServerName=MLFLOW_TRACKING_ARN.split("/")[-1],
    ExpiresInSeconds=300,
)
print("Open this URL in a new tab:")
print(url_resp["AuthorizedUrl"])

### Lab 1: Log a "tweaked" RAG run and compare (15 min)

The whole point of MLflow is **comparison**. Right now you have one run (the Week 18 baseline). Log a second run that pretends you changed something in the RAG pipeline - for example, `top_k_rerank=5` instead of `3`. Use whatever metric values you want; the goal is to see two runs in the UI.

**Steps**:

1. Start a new MLflow run named `week19-rag-tweak-topk5`.
2. Set the tag `source_week` to `19`.
3. Log the same parameter dict as the baseline but override `top_k_rerank=5`.
4. Log metrics: `faithfulness=0.86`, `answer_relevancy=0.81`, `context_precision=0.78`.
5. Store the resulting run id in a variable named `tweaked_run_id` so the rest of the notebook can reference it.
6. Open the MLflow UI (re-run the presigned URL cell if needed) and use the **Compare** button on the two runs.

**Stretch**: Add a third run where `top_k_rerank=1` and worse metrics (say all 0.6). Confirm MLflow's chart shows the trend.

**Homework Extension**: Read the Week 18 RAGAS notebook output for your OWN runs (not the hard-coded baseline). Log those as a separate run with tag `actual_week18=true` and compare to the hard-coded baseline.

In [ ]:
# Lab 1: log a tweaked RAG run so you have something to compare against the baseline.
# Fill in tweaked_params and tweaked_metrics, then open a run, log them, and save its run id.

tweaked_params = None  # YOUR CODE
tweaked_metrics = None  # YOUR CODE
tweaked_run_id = None  # YOUR CODE

with mlflow.start_run(run_name="week19-rag-tweak-topk5") as run:
    pass  # YOUR CODE

In [ ]:
# SAFETY-NET for Lab 1 - run this if you skipped the lab. SKIP if you completed it.
if tweaked_run_id is None:
    print("Using Lab 1 safety-net.")
    with mlflow.start_run(run_name="week19-rag-tweak-topk5") as run:
        mlflow.set_tag("source_week", "19")
        params = dict(week18_pipeline_params)
        params["top_k_rerank"] = 5
        mlflow.log_params(params)
        mlflow.log_metrics({"faithfulness": 0.86, "answer_relevancy": 0.81, "context_precision": 0.78})
        tweaked_run_id = run.info.run_id
    print("Logged tweaked run:", tweaked_run_id)

## Topic 3: SageMaker Training Job + Model Registry

Time to fix the Week 14 problem. You will submit a SageMaker Training Job that fine-tunes DistilBERT on the fraud CSVs you wrote to S3, logs everything to MLflow, and registers the resulting model in the SageMaker Model Registry.

**Important**: a real training job takes 5-10 minutes. To respect class time, your instructor has already run an identical job. You will submit your own job (so you see the API in action) but for the rest of the notebook we will USE THE INSTRUCTOR'S PRE-RUN MODEL ARTIFACT. Look for the comment marked PRE-RUN ARTIFACT.

In [ ]:
# DEMO: train.py - same fine-tuning logic as Week 14, packaged as a SageMaker script.
TRAIN_SCRIPT = r'''
import argparse, os, json
import pandas as pd
import numpy as np
import mlflow
from datasets import Dataset
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          Trainer, TrainingArguments)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "precision": p, "recall": r, "f1": f1}

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--epochs", type=int, default=3)
    parser.add_argument("--learning_rate", type=float, default=2e-5)
    parser.add_argument("--train_batch_size", type=int, default=16)
    parser.add_argument("--model_name", type=str, default="distilbert-base-uncased")
    parser.add_argument("--train_dir", type=str, default=os.environ["SM_CHANNEL_TRAIN"])
    parser.add_argument("--test_dir",  type=str, default=os.environ["SM_CHANNEL_TEST"])
    parser.add_argument("--model_dir", type=str, default=os.environ["SM_MODEL_DIR"])
    args = parser.parse_args()

    def load_csv_dir(d):
        files = [os.path.join(d, f) for f in os.listdir(d) if f.endswith(".csv")]
        return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

    train_df = load_csv_dir(args.train_dir)
    test_df  = load_csv_dir(args.test_dir)

    tokenizer = AutoTokenizer.from_pretrained(args.model_name)
    def tok(b): return tokenizer(b["text"], truncation=True, padding="max_length", max_length=128)
    train_ds = Dataset.from_pandas(train_df).map(tok, batched=True)
    test_ds  = Dataset.from_pandas(test_df).map(tok, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(args.model_name, num_labels=2)
    targs = TrainingArguments(
        output_dir="/opt/ml/checkpoints",
        num_train_epochs=args.epochs,
        learning_rate=args.learning_rate,
        per_device_train_batch_size=args.train_batch_size,
        evaluation_strategy="epoch",
        logging_steps=10,
        save_strategy="no",
        report_to=[],
    )
    trainer = Trainer(model=model, args=targs, train_dataset=train_ds,
                      eval_dataset=test_ds, compute_metrics=compute_metrics)
    trainer.train()
    metrics = trainer.evaluate()
    print("FINAL_METRICS=" + json.dumps(metrics))

    trainer.save_model(args.model_dir)
    tokenizer.save_pretrained(args.model_dir)
    with open(os.path.join(args.model_dir, "metrics.json"), "w") as f:
        json.dump(metrics, f)
'''

with open("/tmp/train.py", "w") as f:
    f.write(TRAIN_SCRIPT)

# Package and upload the source dir
with tarfile.open("/tmp/sourcedir.tar.gz", "w:gz") as tar:
    tar.add("/tmp/train.py", arcname="train.py")

SOURCE_S3 = f"s3://{S3_BUCKET}/{S3_PREFIX}/code/sourcedir.tar.gz"
s3_client.upload_file("/tmp/sourcedir.tar.gz", S3_BUCKET, f"{S3_PREFIX}/code/sourcedir.tar.gz")
print(f"Source uploaded -> {SOURCE_S3}")

In [ ]:
# DEMO: submit a real Training Job. We will NOT wait for it - this is just to
# show the API. The rest of the notebook uses the instructor's PRE-RUN artifact.

HF_IMAGE = ("763104351884.dkr.ecr.us-east-1.amazonaws.com/"
            "huggingface-pytorch-training:2.1.0-transformers4.36.0-gpu-py310-cu118-ubuntu20.04")

job_name = f"week19-distilbert-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"

training_params = {
    "TrainingJobName": job_name,
    "AlgorithmSpecification": {
        "TrainingImage": HF_IMAGE,
        "TrainingInputMode": "File",
    },
    "RoleArn": SAGEMAKER_ROLE_ARN,
    "InputDataConfig": [
        {"ChannelName": "train",
         "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": TRAIN_S3, "S3DataDistributionType": "FullyReplicated"}}},
        {"ChannelName": "test",
         "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": TEST_S3,  "S3DataDistributionType": "FullyReplicated"}}},
    ],
    "OutputDataConfig": {"S3OutputPath": f"s3://{S3_BUCKET}/{S3_PREFIX}/jobs"},
    "ResourceConfig": {"InstanceType": "ml.g4dn.xlarge", "InstanceCount": 1, "VolumeSizeInGB": 30},
    "StoppingCondition": {"MaxRuntimeInSeconds": 1800},
    "HyperParameters": {
        "epochs": "3", "learning_rate": "2e-5", "train_batch_size": "16",
        "sagemaker_program": "train.py",
        "sagemaker_submit_directory": SOURCE_S3,
    },
    "Environment": {
        "HF_TASK": "text-classification",
    },
}

resp = sagemaker_client.create_training_job(**training_params)
print(f"Submitted: {job_name}")
print(f"ARN: {resp['TrainingJobArn']}")
print("Training takes 5-10 minutes. We will NOT wait - moving on to the pre-run artifact.")

In [ ]:
# PRE-RUN ARTIFACT: instructor ran an identical job before class.
# Output lives at this fixed S3 path:
PRETRAINED_MODEL_S3 = f"s3://{S3_BUCKET}/pretrained/model.tar.gz"
PRETRAINED_METRICS = {
    "accuracy": 0.94,
    "precision": 0.91,
    "recall": 0.88,
    "f1": 0.895,
    "eval_loss": 0.18,
}
PRETRAINED_PARAMS = {
    "model_name": "distilbert-base-uncased",
    "epochs": 3,
    "learning_rate": 2e-5,
    "train_batch_size": 16,
    "data_version": DATA_VERSION,
    "train_s3": TRAIN_S3,
}

# Log the pre-run job as a new MLflow run so it appears alongside your RAG runs.
with mlflow.start_run(run_name="week19-distilbert-finetune") as run:
    mlflow.set_tag("source_week", "19")
    mlflow.set_tag("pipeline_type", "classifier")
    mlflow.set_tag("training_job_name", "week19-distilbert-fraud-pretrained")
    mlflow.log_params(PRETRAINED_PARAMS)
    mlflow.log_metrics(PRETRAINED_METRICS)
    mlflow.log_param("model_artifact_s3", PRETRAINED_MODEL_S3)
    TRAINING_RUN_ID = run.info.run_id

print(f"Logged training run: {TRAINING_RUN_ID}")
print(f"Model artifact: {PRETRAINED_MODEL_S3}")

In [ ]:
# DEMO: register the model package in the SageMaker Model Registry

# Step 1: create (or reuse) the Model Package Group
PACKAGE_GROUP = "fraud-classifier-week19"
try:
    sagemaker_client.create_model_package_group(
        ModelPackageGroupName=PACKAGE_GROUP,
        ModelPackageGroupDescription="DistilBERT fraud classifier - Week 19",
    )
    print(f"Created group: {PACKAGE_GROUP}")
except sagemaker_client.exceptions.ClientError as e:
    if "already exists" in str(e):
        print(f"Group {PACKAGE_GROUP} already exists - reusing.")
    else:
        raise

# Step 2: register a new Model Package version pointing at the pre-run artifact
HF_INFERENCE_IMAGE = ("763104351884.dkr.ecr.us-east-1.amazonaws.com/"
                      "huggingface-pytorch-inference:2.1.0-transformers4.36.0-cpu-py310-ubuntu22.04")

resp = sagemaker_client.create_model_package(
    ModelPackageGroupName=PACKAGE_GROUP,
    ModelPackageDescription=f"Trained run {TRAINING_RUN_ID}",
    InferenceSpecification={
        "Containers": [{
            "Image": HF_INFERENCE_IMAGE,
            "ModelDataUrl": PRETRAINED_MODEL_S3,
            "Environment": {"HF_TASK": "text-classification"},
        }],
        "SupportedContentTypes": ["application/json"],
        "SupportedResponseMIMETypes": ["application/json"],
        "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large", "ml.m5.xlarge"],
    },
    ModelApprovalStatus="PendingManualApproval",
    ModelMetrics={
        "ModelQuality": {
            "Statistics": {
                "ContentType": "application/json",
                "S3Uri": f"s3://{S3_BUCKET}/pretrained/metrics.json",
            }
        }
    },
)
MODEL_PACKAGE_ARN = resp["ModelPackageArn"]
print(f"Registered: {MODEL_PACKAGE_ARN}")

### Lab 2: Approve the model and update its description (15 min)

A registered model starts in `PendingManualApproval`. In a real workflow, the data scientist registers it and an ML engineer or governance reviewer approves it after looking at the metrics. You will play both roles.

**Steps**:

1. Call `sagemaker_client.update_model_package(...)` to set `ModelApprovalStatus="Approved"` for `MODEL_PACKAGE_ARN`. Store the response in `approval_response`.
2. Add an `ApprovalDescription` saying "Approved - F1 above 0.85 threshold".
3. Call `sagemaker_client.list_model_packages(ModelPackageGroupName=PACKAGE_GROUP)` and store it in `packages`. Print every package ARN with its approval status.

**Stretch**: Register a SECOND, intentionally worse model package (use a fake worse-metrics S3 file). Reject it with `Rejected` status and a description explaining why.

**Homework Extension**: Read the docs on Model Package Group versioning. Write a 3-line markdown summary of how the registry handles multiple versions of the same group, and what happens when you have both an Approved v1 and a PendingManualApproval v2.

In [ ]:
# Lab 2: approve the registered model package, then confirm the change
approval_response = None  # YOUR CODE

# Confirm by listing all packages in the group
packages = None  # YOUR CODE

In [ ]:
# SAFETY-NET for Lab 2 - run this if you skipped the lab. SKIP if you completed it.
if approval_response is None:
    print("Using Lab 2 safety-net.")
    approval_response = sagemaker_client.update_model_package(
        ModelPackageArn=MODEL_PACKAGE_ARN,
        ModelApprovalStatus="Approved",
        ApprovalDescription="Approved - F1 above 0.85 threshold",
    )
    packages = sagemaker_client.list_model_packages(ModelPackageGroupName=PACKAGE_GROUP)
    for p in packages["ModelPackageSummaryList"]:
        print(p["ModelPackageArn"], "->", p["ModelApprovalStatus"])

## Topic 4: Deploy and plug into the Week 18 supervisor

A model in a registry is not useful until something can call it. You will:

1. Create a SageMaker endpoint from the approved model package.
2. Test the endpoint with `invoke_endpoint`.
3. Wrap that endpoint as a Strands `@tool` called `classify_with_finetuned_model`.
4. Hand the tool to a fresh supervisor that mirrors `week18_supervisor` from last week, plus this new fast-triage tool.

Why bother? In Week 18 every classification decision needed an LLM call - slow and expensive. The fine-tuned DistilBERT runs in ~50 ms on CPU and gives the supervisor a cheap pre-screen.

In [ ]:
# DEMO: create the endpoint from the approved model package
MODEL_NAME      = f"week19-fraud-model-{int(time.time())}"
ENDPOINT_CONFIG = f"week19-fraud-cfg-{int(time.time())}"
ENDPOINT_NAME   = "week19-fraud-endpoint"

# 1) Model object referencing the approved package
sagemaker_client.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=SAGEMAKER_ROLE_ARN,
    Containers=[{"ModelPackageName": MODEL_PACKAGE_ARN}],
)

# 2) Endpoint config
sagemaker_client.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": MODEL_NAME,
        "InstanceType": "ml.m5.large",
        "InitialInstanceCount": 1,
    }],
)

# 3) Endpoint (idempotent - reuse if already deployed by another student today)
try:
    sagemaker_client.create_endpoint(
        EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_CONFIG,
    )
    print(f"Creating endpoint {ENDPOINT_NAME} - this takes 5-7 min the first time only.")
    print("If your instructor has already deployed it today, this errors and we reuse it.")
except sagemaker_client.exceptions.ClientError as e:
    if "already exists" in str(e):
        print(f"Reusing existing endpoint {ENDPOINT_NAME}")
    else:
        raise

# Wait until InService (instructor pre-runs so usually < 30 sec)
waiter = sagemaker_client.get_waiter("endpoint_in_service")
waiter.wait(EndpointName=ENDPOINT_NAME, WaiterConfig={"Delay": 15, "MaxAttempts": 40})
print(f"Endpoint {ENDPOINT_NAME} is InService")

In [ ]:
# DEMO: smoke test the endpoint with a sample transaction
sample_text = "Card-not-present purchase of $4,899 at electronics merchant in country mismatch with billing address"

resp = sagemaker_runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Body=json.dumps({"inputs": sample_text}),
)
prediction = json.loads(resp["Body"].read())
print("Raw response:", prediction)

In [ ]:
# DEMO: wrap the endpoint as a Strands @tool the supervisor can call
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def classify_with_finetuned_model(transaction_description: str) -> str:
    """Fast fraud pre-screen using the Week 19 fine-tuned DistilBERT classifier.

    Returns a JSON string with keys: label ('fraud' or 'legit'), confidence (0-1).
    Use this BEFORE calling slower LLM-based tools to cheaply filter obvious cases.
    """
    resp = sagemaker_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps({"inputs": transaction_description}),
    )
    raw = json.loads(resp["Body"].read())
    # HF text-classification pipeline returns [{"label": "LABEL_0"/"LABEL_1", "score": ...}]
    top = raw[0] if isinstance(raw, list) else raw
    label_map = {"LABEL_0": "legit", "LABEL_1": "fraud"}
    return json.dumps({
        "label": label_map.get(top["label"], top["label"]),
        "confidence": round(float(top["score"]), 4),
    })

# Sanity check
print(classify_with_finetuned_model(sample_text))

### Lab 3: Build the Week 19 supervisor (15 min)

Recreate the Week 18 supervisor pattern, but give it the new `classify_with_finetuned_model` tool alongside the policy retriever. The supervisor's job: for each incoming transaction, call the fast classifier FIRST, and only escalate to the slower KB+rerank flow if confidence is below 0.85.

**Steps**:

1. Create a `BedrockModel` with `model_id="us.anthropic.claude-3-haiku-20240307-v1:0"`.
2. Write a `SYSTEM_PROMPT` that tells the supervisor about the confidence-threshold rule and forces it to call `classify_with_finetuned_model` first.
3. Build an `Agent` (assign it to `week19_supervisor`) whose tools list contains `classify_with_finetuned_model` plus the `policy_retriever_tool` stub provided below.
4. Test the supervisor on both of the provided `test_cases` and print its response.

**Stretch**: Add a third tool that logs every classification to MLflow as a metric (e.g. `mlflow.log_metric("fraud_calls_today", 1)` inside a child run).

**Homework Extension**: Re-run the RAGAS evaluation from Week 18 against your new supervisor, log the scores under a third MLflow run named `week19-supervisor-with-classifier`, and compare to the Week 18 baseline. Did adding the fast classifier change any of the RAG metrics? Write 3 sentences of analysis.

In [ ]:
# Dummy stand-in for the Week 18 retrieval tool (so the lab is self-contained)
@tool
def policy_retriever_tool(query: str) -> str:
    """Stub - returns a canned policy snippet. In real code this is the Week 18 reranked KB retriever."""
    return "Policy: transactions above $5000 from new merchants require manual review."

# Lab 3: build the supervisor
SYSTEM_PROMPT = None  # YOUR CODE
week19_supervisor = None  # YOUR CODE

# Test transactions
test_cases = [
    "Card-not-present purchase of $4,899 at electronics merchant in country mismatch with billing address",
    "Recurring monthly subscription charge of $9.99 to streaming service",
]
for t in test_cases:
    pass  # YOUR CODE

In [ ]:
# SAFETY-NET for Lab 3 - run this if you skipped the lab. SKIP if you completed it.
if week19_supervisor is None:
    print("Using Lab 3 safety-net.")
    SYSTEM_PROMPT = (
        "You are a fraud triage supervisor. For each transaction: "
        "1) ALWAYS call classify_with_finetuned_model first. "
        "2) If confidence >= 0.85, return its decision. "
        "3) If confidence < 0.85, also call policy_retriever_tool and synthesize a final call. "
        "Always include your reasoning."
    )
    model = BedrockModel(model_id="us.anthropic.claude-3-haiku-20240307-v1:0", region_name=AWS_REGION)
    week19_supervisor = Agent(
        model=model,
        system_prompt=SYSTEM_PROMPT,
        tools=[classify_with_finetuned_model, policy_retriever_tool],
    )
    for t in test_cases:
        print("=" * 60)
        print("TXN:", t)
        print(week19_supervisor(t))

## Recap

You closed two open loops from earlier weeks:

- **Week 14**: the un-tracked classifier is now a versioned Model Package with logged hyperparameters, data version, and metrics.
- **Week 18**: the throwaway RAGAS scores are now a logged baseline run you can compare future tweaks against.

You also produced a single MLflow experiment with three runs (RAG baseline, RAG tweak, DistilBERT training) - that experiment is the audit trail Bread Financial's model governance team will ask for.

Next week (Week 20) you will wrap this whole flow in CI/CD with DVC + GitHub Actions, so re-training and re-registering happen on every data-version bump automatically.

## Homework (async)

1. **Tag your runs**: go back to the three MLflow runs you logged and add tags `environment=dev`, `owner=<your name>`. Re-open the UI and use the search bar to filter by tag.
2. **Promote the model**: in the SageMaker console, find your Model Package Group and walk through the "Lineage" view. Take a screenshot of the lineage graph showing data -> training job -> model package -> endpoint.
3. **Read** the SageMaker MLflow docs page on "Compare model versions across multiple runs". Bring one question to next class about a feature you did not understand.
4. **Bonus**: write a one-paragraph proposal for what you would track if Bread Financial's REAL fraud model went into production. What metrics? What alerts? What gates between staging and production?